# 🖥️ Aula 04 — LLM Local no Colab (Ollama + Qwen 3.5 4B)
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula04_ollama_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Professor:** Lucas S. Vieira (luksamuk)

**Objetivo:** Rodar um modelo de linguagem LOCALmente no Colab, usando `requests.post()` — sem API key, sem cloud.

Ao final desse notebook, você vai saber:
- ✅ Instalar e configurar o Ollama no Colab
- ✅ Usar `requests.post()` para conversar com modelos locais
- ✅ Manter um chat com memória (histórico de mensagens)
- ✅ Entender que o padrão HTTP é o mesmo de qualquer API de LLM

**Por que Qwen 3.5 4B?** ~2.7 GB de download, ~80 tok/s na T4, bom custo-benefício para aula.

---


## 1. Verificar GPU

⚠️ **Importante:** Vá em `Runtime → Change runtime type` e selecione **T4 GPU** antes de continuar.

Sem GPU, o Ollama vai rodar na CPU e será muito lento.

In [ ]:
!nvidia-smi

## 2. Instalar Ollama

O Colab não tem algumas dependências que o Ollama precisa. Vamos instalar tudo e configurar a GPU.

In [ ]:
# Instalar dependências + Ollama
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"  # Manter modelo carregado na VRAM

print("✅ Ollama instalado!")

## 3. Iniciar Servidor + Baixar Modelo

O Qwen 3.5 4B tem ~2.7 GB de download. O servidor Ollama roda em background na porta 11434.

In [ ]:
import subprocess, time, requests, os, json

# Matar instância anterior (seguro rodar várias vezes)
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(1)

# Iniciar servidor em background
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ}
)

# Aguardar ficar pronto
print("⏳ Aguardando Ollama iniciar...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("✅ Ollama rodando!")
            break
    except:
        time.sleep(1)

# Baixar modelo (~2.7 GB)
!ollama pull qwen3.5:4b

## 4. Warm Up + Função `chat()`

⚠️ A primeira inferência demora ~2-3 minutos (o modelo carrega na GPU). Depois, respostas são rápidas (~80 tok/s).

Vamos definir uma função `chat()` que usa `requests.post()` — **o padrão HTTP que qualquer API de LLM usa**.

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3.5:4b"

def chat(messages, model=None, stream=False):
    """Envia mensagens pro Ollama e retorna a resposta.

    Args:
        messages: lista [{"role": "...", "content": "..."}]
        model: nome do modelo (padrão: qwen3.5:4b)
        stream: se True, retorna response object para iterar

    Returns:
        dict com a resposta (ou response se stream=True)
    """
    if model is None:
        model = MODEL
    payload = {"model": model, "messages": messages, "stream": stream, "keep_alive": -1}
    response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    if response.status_code != 200:
        raise Exception(f"Erro {response.status_code}: {response.text}")
    return response if stream else response.json()

# Warm up — primeira inferência é lenta!
print("🔥 Aquecendo o modelo (warm up)...")
start = time.time()
r = chat([{"role": "user", "content": "Oi"}])
print(f"✅ Warm up em {time.time()-start:.1f}s")
print(f"   Resposta: {r['message']['content'][:80]}")

## 5. Chat Básico

Agora sim — vamos conversar com o modelo usando `requests.post()` puro, sem SDK!

In [ ]:
# Mensagem simples com system prompt
resultado = chat([
    {"role": "system", "content": "Você é um assistente útil. Responda em português."},
    {"role": "user", "content": "Qual é a capital de Minas Gerais?"}
])
print(f"🤖 {resultado['message']['content']}")
print(f"📊 Tokens: prompt={resultado.get('prompt_eval_count', '?')}, "
      f"completion={resultado.get('eval_count', '?')}")

## 6. Chat com Memória

Cada chamada envia o histórico completo. O modelo "lembra" da conversa porque recebe tudo de novo.

Esse é o mesmo padrão que usamos com dicionários na aula passada!

In [ ]:
# Chat com memória — acumulamos o histórico manualmente
historico = [
    {"role": "system", "content": "Você é um assistente útil. Responda em português. Seja conciso."}
]

historico.append({"role": "user", "content": "Meu nome é Lucas e eu ensino IA."})
r1 = chat(historico)
historico.append({"role": "assistant", "content": r1["message"]["content"]})
print(f"🤖: {r1['message']['content']}")

historico.append({"role": "user", "content": "Qual é o meu nome?"})
r2 = chat(historico)
historico.append({"role": "assistant", "content": r2["message"]["content"]})
print(f"🤖: {r2['message']['content']}")

print(f"\n📝 Histórico tem {len(historico)} mensagens")

## 7. Streaming — Resposta em Tempo Real

Mostra o texto conforme é gerado, igual ao ChatGPT.

In [ ]:
# Streaming — texto aparece palavra por palavra
print("🤖 ", end="")
response = chat(
    [
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "Explique o que é uma API em 3 frases."}
    ],
    stream=True
)
for line in response.iter_lines():
    if line:
        chunk = json.loads(line)
        if "message" in chunk and "content" in chunk["message"]:
            print(chunk["message"]["content"], end="", flush=True)
print()

## 8. API Compatível com OpenAI

O Ollama também expõe `/v1/chat/completions` — formato **idêntico** à OpenAI.

Qualquer código que usa a OpenAI API pode apontar pro Ollama local trocando só a `base_url`.

In [ ]:
OPENAI_URL = "http://localhost:11434/v1/chat/completions"

payload = {
    "model": "qwen3.5:4b",
    "messages": [
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "O que é Python em uma frase?"}
    ],
    "temperature": 0.7,
    "max_tokens": 64,
    "keep_alive": -1
}

response = requests.post(OPENAI_URL, json=payload, timeout=300)
data = response.json()
print(f"🤖 {data['choices'][0]['message']['content']}")
print(f"📊 Model: {data['model']}, Usage: {data['usage']}")

## 9. 🧪 Exercícios

Tente resolver estes exercícios sozinho(a)!

In [ ]:
# Exercício 1: Função universal
# Crie uma função `perguntar()` que funciona com qualquer API de LLM.
# Dica: a função deve aceitar URL, modelo, mensagens e (opcionalmente) API key.

def perguntar(mensagem, url, model, api_key=None, system="Você é um assistente útil."):
    """Faz uma pergunta a qualquer API de LLM usando requests.post()."""
    # Seu código aqui
    pass

# Teste com Ollama:
# resultado = perguntar(
#     "Qual é a capital de MG?",
#     url="http://localhost:11434/v1/chat/completions",
#     model="qwen3.5:4b"
# )
# print(resultado)

In [ ]:
# Exercício 2: Chat interativo
# Crie um loop que lê input do usuário e responde usando a função chat().
# Digite 'sair' para encerrar.

# Seu código aqui:

---

## ✅ Checklist da Aula

- [ ] Sei instalar e configurar o Ollama no Colab
- [ ] Entendo que `requests.post()` é o padrão universal para APIs de LLM
- [ ] Sei criar um chat com memória usando listas de dicionários
- [ ] Sei usar streaming para ver respostas em tempo real
- [ ] Entendo que `/v1/chat/completions` é compatível com a OpenAI

**Ponto-chave:** O padrão é o mesmo — Gemini, Ollama, OpenAI. Só muda a URL e o formato do JSON.
Entender HTTP = entender todas as APIs de LLM! 🚀

---
*Material da Guilda de IA — UFVJM 2026.1*